#### 4. Build a small end-to-end ELT: read CSV, filter, add a column (e.g., ingestion_date), and write the result as Delta.

In [0]:
df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/circuits.csv", header=True, inferSchema = True)

In [0]:
df = df.filter(df.location != 'USA')

In [0]:
from pyspark.sql import functions as F

df = df.withColumn('ingestion_date', F.col('_metadata.file_modification_time'))

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('cyntexa_dev.sales.circuits_via_spark')

#### 5. Read a JSON file with nested structure and flatten at least one nested field using dot notation or explode().

In [0]:
jsondf = spark.read.json('/Volumes/cyntexa_dev/sales/raw/drivers.json')

jsondf.select('code', 'name.forename', 'name.surname', 'nationality', 'dob', 'url', 'driverId', 'driverRef', 'number', 'url').display()

#### 6. Call .explain() on a multi-step transformation chain and identify, from the physical plan, which steps got pipelined together versus which required a shuffle.

In [0]:
import pyspark.sql.functions as F

jsondf = jsondf.withColumn('nationality', F.upper(F.col('nationality')))
jsondf = jsondf.withColumnsRenamed({'name.forename': 'first_name', 'name.surname': 'last_name'})
jsondf = jsondf.withColumn('driverRef', F.concat(F.upper(F.substring(F.col('driverRef'), 1, 1)), F.substring(F.col('driverRef'), 2, 100)))
jsondf = jsondf.withColumn('dob', F.to_date(F.col('dob'), 'dd-MM-yyyy'))

jsondf.explain()


== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [code#11347, cast(gettimestamp(dob#11348, dd-MM-yyyy, TimestampType, try_to_date, Some(Etc/UTC), true) as date) AS dob#11369, driverId#11349L, concat(upper(ephemeralsubstring(driverRef#11350, 1, 1)), substring(driverRef#11350, 2, 100)) AS driverRef#11367, name#11351, upper(nationality#11352) AS nationality#11365, number#11353, url#11354]
      +- PhotonJsonScan json [code#11347,dob#11348,driverId#11349L,driverRef#11350,name#11351,nationality#11352,number#11353,url#11354] Batched: true, DataFilters: [], Format: JSON, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/cyntexa_dev/sales/raw/drivers.json], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<code:string,dob:string,driverId:bigint,driverRef:string,name:struct<forename:string,surnam...


== Photon Explanation ==
The query is fully supported by Photon.